# 6교시. OCR 및 정보 추출 기능 연동

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/06_ocr_ai_integration.ipynb)

**이번 교시 행동:** 업로드한 파일을 실제 OCR 함수에 연결하고 LIVE·오류·복구 모드를 화면에서 구분합니다.

**통과 증거:** `course_outputs/app_06.py`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 실행 모드를 먼저 확인합니다.

- `LIVE`: 현재 파일에 실제 모델을 실행한 결과
- `PREPARED_FALLBACK`: 공개 샘플을 사람이 검수해 둔 복구 결과
- 3분 이상 멈추면 실행을 중지하고 복구 결과로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


## 코드 셀을 읽는 방법

각 코드 셀의 맨 위에는 `코드 읽기` 주석이 있습니다.

1. `수정하지 않습니다`라고 적힌 셀은 설명을 읽고 그대로 실행합니다.
2. `TODO`가 있는 셀은 안내된 `None` 또는 짧은 값만 바꿉니다.
3. 실행 출력에서 `코드 읽는 법`과 `확인할 결과`를 다시 확인합니다.
4. `단계 실행 완료`가 나온 뒤 다음 코드 셀로 이동합니다.

Python 문법 전체를 먼저 이해할 필요는 없습니다. 변수에 어떤 값이 들어가고,
실행 뒤 어떤 결과가 달라지는지를 중심으로 읽습니다.


In [ ]:
def _show_learning_message(markdown_text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown_text))
    except ImportError:
        print(markdown_text)


def show_lab_step(current, total, title, action, expected, code_help):
    _show_learning_message(
        f"""---
### 🧪 실습 단계 {current}/{total} · {title}

**지금 할 일:** {action}

**코드 읽는 법:** {code_help}

**이 단계에서 확인할 결과:** {expected}
"""
    )


def complete_lab_step(current, total, expected):
    next_action = (
        "결과를 확인한 뒤 다음 코드 셀을 실행하세요."
        if current < total
        else "마지막 CHECKPOINT와 산출물 파일을 확인하세요."
    )
    _show_learning_message(
        f"""> ✅ **{current}/{total} 단계 실행 완료**
>
> **결과 확인:** {expected}
>
> **다음 행동:** {next_action}
"""
    )

# ── 코드 읽기 ─────────────────────────────────────────────
# OCR 연동 앱과 결과 파일을 위한 공통 폴더·자료 로더를 준비합니다. 설정 코드이므로 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(1, 8, '공통 환경 준비', '연동 앱 파일과 실습 자료를 위한 공통 환경을 준비합니다.', 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.', 'OCR 연동 앱과 결과 파일을 위한 공통 폴더·자료 로더를 준비합니다. 설정 코드이므로 수정하지 않습니다.')

import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_PREPARED") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_PREPARED_INPUT=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/document_ai_lecture_2026/"
)

def load_course_assets(*relative_paths):
    if VALIDATION_MODE:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded

complete_lab_step(1, 8, 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# Streamlit과 PaddleOCR의 고정 버전을 확인하고 필요한 경우에만 설치합니다. 설치 실패 시 준비 결과로 계속할 수 있도록 실패 이유를
# 보존합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(2, 8, 'Streamlit 준비', '실습에 고정한 Streamlit 버전을 확인하고 필요하면 설치합니다.', '오류 없이 끝나면 웹앱 실행 환경이 준비된 것입니다.', 'Streamlit과 PaddleOCR의 고정 버전을 확인하고 필요한 경우에만 설치합니다. 설치 실패 시 준비 결과로 계속할 수 있도록 실패 이유를 보존합니다.')

import importlib.metadata
import subprocess

required_streamlit = "1.60.0"
try:
    installed_streamlit = importlib.metadata.version("streamlit")
except importlib.metadata.PackageNotFoundError:
    installed_streamlit = None
if installed_streamlit != required_streamlit:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", f"streamlit=={required_streamlit}"]
    )

required_paddlepaddle = "3.2.1"
required_paddleocr = "3.7.0"
if VALIDATION_MODE:
    PADDLEOCR_READY = False
    print("자동검증 모드: PaddleOCR 설치를 생략합니다.")
else:
    try:
        installed_paddlepaddle = importlib.metadata.version("paddlepaddle")
        installed_paddleocr = importlib.metadata.version("paddleocr")
        if (
            installed_paddlepaddle != required_paddlepaddle
            or installed_paddleocr != required_paddleocr
        ):
            raise importlib.metadata.PackageNotFoundError
        PADDLEOCR_READY = True
    except importlib.metadata.PackageNotFoundError:
        try:
            subprocess.check_call(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    f"paddlepaddle=={required_paddlepaddle}",
                    f"paddleocr=={required_paddleocr}",
                ]
            )
            PADDLEOCR_READY = True
        except Exception as exc:
            PADDLEOCR_READY = False
            print("LIVE OCR 준비 실패:", type(exc).__name__, exc)

print("Streamlit:", importlib.metadata.version("streamlit"))
print(
    "PaddleOCR LIVE:",
    (
        importlib.metadata.version("paddleocr")
        if PADDLEOCR_READY
        else "준비 실패 · PREPARED_FALLBACK 사용"
    ),
)

complete_lab_step(2, 8, '오류 없이 끝나면 웹앱 실행 환경이 준비된 것입니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `app_code`에는 업로드 파일을 OCR 함수에 전달하는 전체 앱 코드가 있습니다. `write_text()`가 이를 `app_06.py`로
# 저장합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(3, 8, 'OCR 연동 앱 생성', 'LIVE OCR·오류·복구 모드를 가진 앱 파일을 저장합니다.', '`app_06.py` 저장 경로가 표시되어야 합니다.', '`app_code`에는 업로드 파일을 OCR 함수에 전달하는 전체 앱 코드가 있습니다. `write_text()`가 이를 `app_06.py`로 저장합니다.')

app_code = (
    'import tempfile\n'
    'from pathlib import Path\n'
    'import streamlit as st\n'
    '\n'
    "GOLDEN_OCR_TEXT = '이태리집\\n거래일시 2025-10-04 12:33:37\\n페퍼로니 앤 치즈 29,000 1 29,000\\n토마토 파스타 14,000 1 14,000\\n수제 돈가스 13,000 1 13,000\\n새우 칠리치 필라 14,000 1 14,000\\n콜라 2,000 3 6,000\\n합계 금액 76,000\\n부가세 과세물품가액 69,094\\n부가세 6,906\\n'\n"
    "GOLDEN_RECEIPT = {'document_type': 'receipt',\n"
    " 'store_name': '이태리집',\n"
    " 'date': '2025-10-04',\n"
    " 'total_amount': 76000,\n"
    " 'items': [{'name': '페퍼로니 앤 치즈',\n"
    "            'quantity': 1,\n"
    "            'unit_price': 29000,\n"
    "            'line_total': 29000},\n"
    "           {'name': '토마토 파스타', 'quantity': 1, 'unit_price': 14000, 'line_total': 14000},\n"
    "           {'name': '수제 돈가스', 'quantity': 1, 'unit_price': 13000, 'line_total': 13000},\n"
    "           {'name': '새우 칠리치 필라',\n"
    "            'quantity': 1,\n"
    "            'unit_price': 14000,\n"
    "            'line_total': 14000},\n"
    "           {'name': '콜라', 'quantity': 3, 'unit_price': 2000, 'line_total': 6000}],\n"
    " 'adjustments': {'discount': 0, 'tax': 0, 'service': 0, 'rounding': 0},\n"
    " 'tax_breakdown': {'mode': 'included_in_item_prices',\n"
    "                   'supply_amount': 69094,\n"
    "                   'vat': 6906,\n"
    "                   'payable_total': 76000},\n"
    " 'raw_values': {'store_name': '이태리집',\n"
    "                'date': '2025-10-04 12:33:37',\n"
    "                'total_amount': '76,000'},\n"
    " 'cleaned_values': {'store_name': '이태리집', 'date': '2025-10-04', 'total_amount': 76000},\n"
    " 'evidence': {'store_name': {'raw_value': '이태리집', 'line': 1},\n"
    "              'date': {'raw_value': '거래일시 2025-10-04 12:33:37', 'line': 2},\n"
    "              'total_amount': {'raw_value': '합계 금액 76,000', 'line': 8}},\n"
    " 'source_mode': 'prepared_fixture_rule_extraction'}\n"
    '\n'
    'from collections import defaultdict\n'
    '\n'
    'def reconstruct_spatial_lines(items):\n'
    '    positioned_by_page = defaultdict(list)\n'
    '    unpositioned_by_page = defaultdict(list)\n'
    '    for order, item in enumerate(items):\n'
    '        text = " ".join(str(item.get("text", "")).split())\n'
    '        if not text:\n'
    '            continue\n'
    '        page = int(item.get("page") or 1)\n'
    '        points = [\n'
    '            point\n'
    '            for point in (item.get("box") or [])\n'
    '            if isinstance(point, (list, tuple)) and len(point) >= 2\n'
    '        ]\n'
    '        if not points:\n'
    '            unpositioned_by_page[page].append((order, text))\n'
    '            continue\n'
    '        xs = [float(point[0]) for point in points]\n'
    '        ys = [float(point[1]) for point in points]\n'
    '        positioned_by_page[page].append({\n'
    '            "text": text,\n'
    '            "x": min(xs),\n'
    '            "y": sum(ys) / len(ys),\n'
    '            "height": max(ys) - min(ys),\n'
    '            "order": order,\n'
    '        })\n'
    '\n'
    '    pages = sorted(set(positioned_by_page) | set(unpositioned_by_page))\n'
    '    lines = []\n'
    '    for page in pages:\n'
    '        rows = []\n'
    '        for token in sorted(\n'
    '            positioned_by_page[page],\n'
    '            key=lambda value: (value["y"], value["x"], value["order"]),\n'
    '        ):\n'
    '            row = rows[-1] if rows else None\n'
    '            tolerance = (\n'
    '                max(12.0, min(24.0, max(row["height"], token["height"]) * 0.45))\n'
    '                if row else 12.0\n'
    '            )\n'
    '            if row and abs(token["y"] - row["y"]) <= tolerance:\n'
    '                row["tokens"].append(token)\n'
    '                count = len(row["tokens"])\n'
    '                row["y"] = (row["y"] * (count - 1) + token["y"]) / count\n'
    '                row["height"] = max(row["height"], token["height"])\n'
    '            else:\n'
    '                rows.append({\n'
    '                    "tokens": [token],\n'
    '                    "y": token["y"],\n'
    '                    "height": token["height"],\n'
    '                })\n'
    '\n'
    '        lines.extend(\n'
    '            " ".join(\n'
    '                token["text"]\n'
    '                for token in sorted(\n'
    '                    row["tokens"],\n'
    '                    key=lambda value: (value["x"], value["order"]),\n'
    '                )\n'
    '            )\n'
    '            for row in rows\n'
    '        )\n'
    '        lines.extend(\n'
    '            text\n'
    '            for _, text in sorted(\n'
    '                unpositioned_by_page[page],\n'
    '                key=lambda value: value[0],\n'
    '            )\n'
    '        )\n'
    '    return lines\n'
    '\n'
    'import re\n'
    '\n'
    'def to_int(value):\n'
    '    return int(value.replace(",", ""))\n'
    '\n'
    '\n'
    'def extract_receipt_from_text(text, source_mode):\n'
    '    lines = [line.strip() for line in text.splitlines() if line.strip()]\n'
    '    date_match = re.search(r"\\b(\\d{4})[-./](\\d{1,2})[-./](\\d{1,2})\\b", text)\n'
    '    total_line = next(\n'
    '        (\n'
    '            line\n'
    '            for line in lines\n'
    '            if re.search(r"(?:합\\s*계|결제\\s*금액|총\\s*액)", line)\n'
    '        ),\n'
    '        None,\n'
    '    )\n'
    '    total_candidates = (\n'
    '        re.findall(r"(?<![\\d,])\\d[\\d,]*(?![\\d,])", total_line)\n'
    '        if total_line\n'
    '        else []\n'
    '    )\n'
    '    total_raw = total_candidates[-1] if total_candidates else None\n'
    '    supply_match = re.search(\n'
    '        r"(?:부가세\\s*)?과세물품가액\\s*[:：]?\\s*([\\d,]+)",\n'
    '        text,\n'
    '    )\n'
    '    vat_match = re.search(\n'
    '        r"^부가세(?!\\s*과세물품가액)\\s*[:：]?\\s*([\\d,]+)",\n'
    '        text,\n'
    '        re.MULTILINE,\n'
    '    )\n'
    '    item_pattern = re.compile(\n'
    '        r"^(?P<name>.+?)\\s+(?P<unit>[\\d,]+)\\s+"\n'
    '        r"(?P<quantity>\\d+)\\s+(?P<line>[\\d,]+)$"\n'
    '    )\n'
    '    markdown_item_pattern = re.compile(\n'
    '        r"^\\|\\s*(?P<name>[^|]+?)\\s*\\|\\s*(?P<quantity>\\d+)\\s*\\|"\n'
    '        r"\\s*(?P<unit>[\\d,]+)원\\s*\\|\\s*(?P<line>[\\d,]+)원\\s*\\|$"\n'
    '    )\n'
    '    items = []\n'
    '    item_evidence = []\n'
    '    for line_number, line in enumerate(lines, start=1):\n'
    '        match = item_pattern.search(line)\n'
    '        if not match:\n'
    '            match = markdown_item_pattern.search(line)\n'
    '        if match:\n'
    '            item = {\n'
    '                "name": match.group("name"),\n'
    '                "quantity": int(match.group("quantity")),\n'
    '                "unit_price": to_int(match.group("unit")),\n'
    '                "line_total": to_int(match.group("line")),\n'
    '            }\n'
    '            items.append(item)\n'
    '            item_evidence.append({"line": line_number, "raw_value": line})\n'
    '\n'
    '    date_value = (\n'
    '        f"{int(date_match.group(1)):04d}-{int(date_match.group(2)):02d}-"\n'
    '        f"{int(date_match.group(3)):02d}"\n'
    '        if date_match else None\n'
    '    )\n'
    '    total_value = to_int(total_raw) if total_raw else None\n'
    '    supply_value = to_int(supply_match.group(1)) if supply_match else None\n'
    '    vat_value = to_int(vat_match.group(1)) if vat_match else None\n'
    '    return {\n'
    '        "document_type": "receipt",\n'
    '        "store_name": lines[0] if lines else None,\n'
    '        "date": date_value,\n'
    '        "total_amount": total_value,\n'
    '        "items": items,\n'
    '        "adjustments": {"discount": 0, "tax": 0, "service": 0, "rounding": 0},\n'
    '        "tax_breakdown": {\n'
    '            "mode": "included_in_item_prices",\n'
    '            "supply_amount": supply_value,\n'
    '            "vat": vat_value,\n'
    '            "payable_total": total_value,\n'
    '        } if supply_value is not None and vat_value is not None else None,\n'
    '        "raw_values": {\n'
    '            "store_name": lines[0] if lines else None,\n'
    '            "date": date_match.group(0) if date_match else None,\n'
    '            "total_amount": total_raw,\n'
    '        },\n'
    '        "cleaned_values": {\n'
    '            "store_name": lines[0] if lines else None,\n'
    '            "date": date_value,\n'
    '            "total_amount": total_value,\n'
    '        },\n'
    '        "evidence": {\n'
    '            "store_name": {"line": 1, "raw_value": lines[0] if lines else None},\n'
    '            "date": {"raw_value": date_match.group(0) if date_match else None},\n'
    '            "total_amount": {"raw_value": total_line},\n'
    '            "items": item_evidence,\n'
    '        },\n'
    '        "source_mode": source_mode,\n'
    '    }\n'
    '\n'
    'def run_live_ocr(uploaded):\n'
    '    suffix = Path(uploaded.name).suffix.lower()\n'
    '    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as temp:\n'
    '        temp.write(uploaded.getvalue())\n'
    '        path = temp.name\n'
    '    try:\n'
    '        from paddleocr import PaddleOCR\n'
    '        engine = PaddleOCR(\n'
    '            lang="korean",\n'
    '            ocr_version="PP-OCRv5",\n'
    '            use_doc_orientation_classify=False,\n'
    '            use_doc_unwarping=False,\n'
    '            use_textline_orientation=False,\n'
    '            device="cpu",\n'
    '        )\n'
    '        page = list(engine.predict(path))[0]\n'
    '        payload = page.json() if callable(page.json) else page.json\n'
    '        result = payload.get("res", payload)\n'
    '        items = [\n'
    '            {\n'
    '                "page": 1,\n'
    '                "box": box.tolist() if hasattr(box, "tolist") else box,\n'
    '                "text": text,\n'
    '                "confidence": float(score),\n'
    '            }\n'
    '            for box, text, score in zip(\n'
    '                result.get("rec_polys", []),\n'
    '                result.get("rec_texts", []),\n'
    '                result.get("rec_scores", []),\n'
    '            )\n'
    '        ]\n'
    '        return "\\n".join(reconstruct_spatial_lines(items))\n'
    '    finally:\n'
    '        Path(path).unlink(missing_ok=True)\n'
    '\n'
    '\n'
    'def process_document(uploaded=None, *, use_prepared=False):\n'
    '    if use_prepared:\n'
    '        text = GOLDEN_OCR_TEXT\n'
    '        mode = "PREPARED_FALLBACK"\n'
    '    elif uploaded is None:\n'
    '        return {"ok": False, "mode": "INPUT_ERROR", "error": "파일을 선택하세요."}\n'
    '    else:\n'
    '        try:\n'
    '            text = run_live_ocr(uploaded)\n'
    '            mode = "LIVE"\n'
    '        except Exception as exc:\n'
    '            return {\n'
    '                "ok": False,\n'
    '                "mode": "LIVE_ERROR",\n'
    '                "error": f"{type(exc).__name__}: {exc}",\n'
    '                "recovery": "공개 샘플 준비 결과 버튼을 선택하세요.",\n'
    '            }\n'
    '    data = extract_receipt_from_text(\n'
    '        text,\n'
    '        "live_ocr_rule_extraction" if mode == "LIVE"\n'
    '        else "prepared_fixture_rule_extraction",\n'
    '    )\n'
    '    return {"ok": True, "mode": mode, "ocr_text": text, "data": data}\n'
    '\n'
    '\n'
    'st.title("영수증 Document AI 연결 앱")\n'
    'uploaded = st.file_uploader(\n'
    '    "승인된 비식별 이미지 또는 PDF 한 장 · 최대 5MB",\n'
    '    type=["png", "jpg", "jpeg", "pdf"],\n'
    '    max_upload_size=5,\n'
    '    help="PNG, JPG, JPEG, PDF만 허용합니다. 수업에서는 한 번에 5MB 이하 한 장만 처리합니다.",\n'
    ')\n'
    'left, right = st.columns(2)\n'
    'run_live = left.button("업로드 파일 LIVE 처리")\n'
    'run_prepared = right.button("공개 샘플 준비 결과")\n'
    'if run_live or run_prepared:\n'
    '    result = process_document(uploaded, use_prepared=run_prepared)\n'
    '    if result["ok"]:\n'
    '        st.success(f"실행 모드: {result[\'mode\']}")\n'
    '        st.text_area("OCR 원문", result["ocr_text"], height=220)\n'
    '        st.json(result["data"])\n'
    '    else:\n'
    '        st.error(f"{result[\'mode\']} · {result[\'error\']}")\n'
    '        if result.get("recovery"):\n'
    '            st.info(result["recovery"])\n'
)

output_path = OUTPUT_DIR / "app_06.py"
output_path.write_text(app_code, encoding="utf-8")
print("저장:", output_path)

complete_lab_step(3, 8, '`app_06.py` 저장 경로가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `reconstruct_spatial_lines()`가 실제 OCR 좌표를 행으로 복원한 뒤 `extract_receipt_from_text()`
# 결과의 날짜·총액·품목 수를 검사합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(4, 8, '실제 OCR 기록 회귀검사', '보존한 PP-OCRv5 좌표를 행으로 복원하고 추출값을 검사합니다.', '`RECORDED LIVE REGRESSION PASS: 76000 5`를 확인합니다.', '`reconstruct_spatial_lines()`가 실제 OCR 좌표를 행으로 복원한 뒤 `extract_receipt_from_text()` 결과의 날짜·총액·품목 수를 검사합니다.')

from collections import defaultdict

def reconstruct_spatial_lines(items):
    positioned_by_page = defaultdict(list)
    unpositioned_by_page = defaultdict(list)
    for order, item in enumerate(items):
        text = " ".join(str(item.get("text", "")).split())
        if not text:
            continue
        page = int(item.get("page") or 1)
        points = [
            point
            for point in (item.get("box") or [])
            if isinstance(point, (list, tuple)) and len(point) >= 2
        ]
        if not points:
            unpositioned_by_page[page].append((order, text))
            continue
        xs = [float(point[0]) for point in points]
        ys = [float(point[1]) for point in points]
        positioned_by_page[page].append({
            "text": text,
            "x": min(xs),
            "y": sum(ys) / len(ys),
            "height": max(ys) - min(ys),
            "order": order,
        })

    pages = sorted(set(positioned_by_page) | set(unpositioned_by_page))
    lines = []
    for page in pages:
        rows = []
        for token in sorted(
            positioned_by_page[page],
            key=lambda value: (value["y"], value["x"], value["order"]),
        ):
            row = rows[-1] if rows else None
            tolerance = (
                max(12.0, min(24.0, max(row["height"], token["height"]) * 0.45))
                if row else 12.0
            )
            if row and abs(token["y"] - row["y"]) <= tolerance:
                row["tokens"].append(token)
                count = len(row["tokens"])
                row["y"] = (row["y"] * (count - 1) + token["y"]) / count
                row["height"] = max(row["height"], token["height"])
            else:
                rows.append({
                    "tokens": [token],
                    "y": token["y"],
                    "height": token["height"],
                })

        lines.extend(
            " ".join(
                token["text"]
                for token in sorted(
                    row["tokens"],
                    key=lambda value: (value["x"], value["order"]),
                )
            )
            for row in rows
        )
        lines.extend(
            text
            for _, text in sorted(
                unpositioned_by_page[page],
                key=lambda value: value[0],
            )
        )
    return lines


import re

def to_int(value):
    return int(value.replace(",", ""))


def extract_receipt_from_text(text, source_mode):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    date_match = re.search(r"\b(\d{4})[-./](\d{1,2})[-./](\d{1,2})\b", text)
    total_line = next(
        (
            line
            for line in lines
            if re.search(r"(?:합\s*계|결제\s*금액|총\s*액)", line)
        ),
        None,
    )
    total_candidates = (
        re.findall(r"(?<![\d,])\d[\d,]*(?![\d,])", total_line)
        if total_line
        else []
    )
    total_raw = total_candidates[-1] if total_candidates else None
    supply_match = re.search(
        r"(?:부가세\s*)?과세물품가액\s*[:：]?\s*([\d,]+)",
        text,
    )
    vat_match = re.search(
        r"^부가세(?!\s*과세물품가액)\s*[:：]?\s*([\d,]+)",
        text,
        re.MULTILINE,
    )
    item_pattern = re.compile(
        r"^(?P<name>.+?)\s+(?P<unit>[\d,]+)\s+"
        r"(?P<quantity>\d+)\s+(?P<line>[\d,]+)$"
    )
    markdown_item_pattern = re.compile(
        r"^\|\s*(?P<name>[^|]+?)\s*\|\s*(?P<quantity>\d+)\s*\|"
        r"\s*(?P<unit>[\d,]+)원\s*\|\s*(?P<line>[\d,]+)원\s*\|$"
    )
    items = []
    item_evidence = []
    for line_number, line in enumerate(lines, start=1):
        match = item_pattern.search(line)
        if not match:
            match = markdown_item_pattern.search(line)
        if match:
            item = {
                "name": match.group("name"),
                "quantity": int(match.group("quantity")),
                "unit_price": to_int(match.group("unit")),
                "line_total": to_int(match.group("line")),
            }
            items.append(item)
            item_evidence.append({"line": line_number, "raw_value": line})

    date_value = (
        f"{int(date_match.group(1)):04d}-{int(date_match.group(2)):02d}-"
        f"{int(date_match.group(3)):02d}"
        if date_match else None
    )
    total_value = to_int(total_raw) if total_raw else None
    supply_value = to_int(supply_match.group(1)) if supply_match else None
    vat_value = to_int(vat_match.group(1)) if vat_match else None
    return {
        "document_type": "receipt",
        "store_name": lines[0] if lines else None,
        "date": date_value,
        "total_amount": total_value,
        "items": items,
        "adjustments": {"discount": 0, "tax": 0, "service": 0, "rounding": 0},
        "tax_breakdown": {
            "mode": "included_in_item_prices",
            "supply_amount": supply_value,
            "vat": vat_value,
            "payable_total": total_value,
        } if supply_value is not None and vat_value is not None else None,
        "raw_values": {
            "store_name": lines[0] if lines else None,
            "date": date_match.group(0) if date_match else None,
            "total_amount": total_raw,
        },
        "cleaned_values": {
            "store_name": lines[0] if lines else None,
            "date": date_value,
            "total_amount": total_value,
        },
        "evidence": {
            "store_name": {"line": 1, "raw_value": lines[0] if lines else None},
            "date": {"raw_value": date_match.group(0) if date_match else None},
            "total_amount": {"raw_value": total_line},
            "items": item_evidence,
        },
        "source_mode": source_mode,
    }


OCR_RECORD_PATH = "tests/fixtures/ppocrv5_live_receipt_tokens.json"
recorded_asset = load_course_assets(OCR_RECORD_PATH)
RECORDED_PP_OCRV5_TOKENS = json.loads(
    recorded_asset[OCR_RECORD_PATH].decode("utf-8")
)
recorded_text = "\n".join(
    reconstruct_spatial_lines(RECORDED_PP_OCRV5_TOKENS)
)
recorded_receipt = extract_receipt_from_text(
    recorded_text,
    "recorded_ppocrv5_regression",
)
assert recorded_receipt["date"] == "2025-10-04"
assert recorded_receipt["total_amount"] == 76000
assert len(recorded_receipt["items"]) == 5
print(
    "RECORDED LIVE REGRESSION PASS:",
    recorded_receipt["total_amount"],
    len(recorded_receipt["items"]),
)

complete_lab_step(4, 8, '`RECORDED LIVE REGRESSION PASS: 76000 5`를 확인합니다.')


## 내가 직접 정하는 LIVE 통과 조건 3개

앱이 오류 없이 열리는 것과 추출값이 맞는 것은 다릅니다. LIVE 경로가
반드시 확인해야 할 값을 세 개 고릅니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `my_live_checks`의 세 `None`만 필드명으로 바꿉니다. 앱이 열리는 것과 추출값이 맞는 것은 다르므로 업무 통과 조건을 고릅니다.
# ──────────────────────────────────────────────────────────
show_lab_step(5, 8, '내 LIVE 통과 조건 입력', 'LIVE 결과에서 반드시 맞아야 할 필드 세 개를 정합니다.', '빈칸 안내 또는 내가 입력한 세 필드가 표시되어야 합니다.', '`my_live_checks`의 세 `None`만 필드명으로 바꿉니다. 앱이 열리는 것과 추출값이 맞는 것은 다르므로 업무 통과 조건을 고릅니다.')

# TODO: 확인할 필드 세 개를 채우세요.
my_live_checks = [None, None, None]
if any(value is None for value in my_live_checks):
    print("빈칸이 있습니다. 아래 전체 정답과 비교하세요.")

complete_lab_step(5, 8, '빈칸 안내 또는 내가 입력한 세 필드가 표시되어야 합니다.')


<details>
<summary>힌트와 전체 정답 보기</summary>

날짜, 총액, 반복 품목 수는 후속 검증과 Excel에 직접 영향을 줍니다.
</details>


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `ANSWER_LIVE_CHECKS`는 날짜·총액·품목을 필수 확인값으로 제시합니다. 내 답과 비교하고 수정 없이 실행합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(6, 8, 'LIVE 통과 조건 정답 확인', '날짜·총액·품목을 필수 확인값으로 확정합니다.', '세 필드가 담긴 전체 정답을 확인합니다.', '`ANSWER_LIVE_CHECKS`는 날짜·총액·품목을 필수 확인값으로 제시합니다. 내 답과 비교하고 수정 없이 실행합니다.')

ANSWER_LIVE_CHECKS = ["date", "total_amount", "items"]
assert recorded_receipt["date"]
assert recorded_receipt["total_amount"] is not None
assert recorded_receipt["items"]
print("전체 정답 · LIVE 필수 확인:", ANSWER_LIVE_CHECKS)

complete_lab_step(6, 8, '세 필드가 담긴 전체 정답을 확인합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `AppTest`가 준비 결과 버튼을 누르고 실행 모드와 JSON이 화면에 나오는지 검사합니다. 실제 OCR 정확도 검사는 앞의 회귀 셀이
# 담당합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(7, 8, '연동 앱 자동 동작 검사', '준비 결과 버튼이 모드와 JSON을 화면에 표시하는지 검사합니다.', '`CHECKPOINT 1/1 PASS`가 표시되어야 합니다.', '`AppTest`가 준비 결과 버튼을 누르고 실행 모드와 JSON이 화면에 나오는지 검사합니다. 실제 OCR 정확도 검사는 앞의 회귀 셀이 담당합니다.')

from streamlit.testing.v1 import AppTest

app_test = AppTest.from_file(str(output_path)).run(timeout=20)
assert not app_test.exception
assert app_test.title[0].value == "영수증 Document AI 연결 앱"
assert len(app_test.button) == 2
app_test.button[1].click().run(timeout=20)
assert any("PREPARED_FALLBACK" in item.value for item in app_test.success)
assert app_test.json
print("CHECKPOINT 1/1 PASS: 앱 연결·모드 표시·JSON 출력")

complete_lab_step(7, 8, '`CHECKPOINT 1/1 PASS`가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# Streamlit 서버를 Colab iframe으로 열어 LIVE와 복구 버튼을 직접 조작합니다. 서버 프로세스와 자동검증 분기를 수정하지
# 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(8, 8, '연동 앱 직접 조작', 'Colab 안에서 LIVE 처리와 준비 결과 버튼을 직접 조작합니다.', '앱 화면 또는 검증 모드 생략 안내를 확인합니다.', 'Streamlit 서버를 Colab iframe으로 열어 LIVE와 복구 버튼을 직접 조작합니다. 서버 프로세스와 자동검증 분기를 수정하지 않습니다.')

# 선택 실습 · 녹화에서는 이 셀로 실제 화면을 엽니다.
# AppTest가 필수 검증이며, 미리보기에는 공개 비식별 샘플만 사용합니다.
if not VALIDATION_MODE:
    import subprocess
    import time
    import urllib.request

    preview_process = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run",
            str(output_path),
            "--server.port", "8506",
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    for _ in range(20):
        try:
            urllib.request.urlopen(
                "http://127.0.0.1:8506/_stcore/health",
                timeout=1,
            )
            break
        except Exception:
            time.sleep(0.5)
    try:
        from google.colab import output
        print("아래 화면에서 직접 버튼과 입력값을 조작하세요.")
        output.serve_kernel_port_as_iframe(8506, height=760)
    except Exception as exc:
        print("Colab 미리보기를 열지 못했습니다:", exc)
        print("AppTest 결과와 app 파일로 계속합니다.")
else:
    print("검증 모드: 대화형 Streamlit 미리보기 생략")

complete_lab_step(8, 8, '앱 화면 또는 검증 모드 생략 안내를 확인합니다.')
